# 🌿 Huấn luyện Mô hình TinyML Phát hiện Dị thường Cảm biến DHT20 cho ESP32-S3
**Môn học:** Internet of Things (CO3061 / CO3062) — ĐH Bách Khoa TP.HCM (HCMUT)  
**Giảng viên hướng dẫn:** Thầy Lê Trọng Nhân  
**Sinh viên thực hiện:** Phạm Công Võ (MSSV: 2313946 — ID: `vophamk23`)  

---
### 📋 Mục tiêu của Notebook này:
1. **Tạo tập dữ liệu huấn luyện (Dataset Generation):** Giả lập và gán nhãn các trạng thái vi khí hậu nhà kính (Bình thường vs Bất thường / Hơ lửa / Khô hạn / Ẩm bão hòa).
2. **Xây dựng Mạng nơ-ron (MLP):** Mạng nơ-ron 2 lớp siêu nhẹ (2 inputs → 8 hidden neurons → 1 output probability).
3. **Huấn luyện & Đánh giá (Training & Evaluation):** Đo độ chính xác (Accuracy), Ma trận nhầm lẫn (Confusion Matrix) và đường biên phân lớp.
4. **Lượng tử hóa & Chuyển đổi sang TensorFlow Lite (`.tflite`):** Tối ưu kích thước mô hình xuống dưới 1 KB.
5. **Xuất file Header C++ (`dht_anomaly_model.h`):** Tự động sinh mảng `const unsigned char dht_anomaly_model_tflite[]` để nạp trực tiếp vào firmware ESP32-S3!

## Bước 1: Import các thư viện cần thiết

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

print(f"TensorFlow Version: {tf.__version__}")
print(f"NumPy Version: {np.__version__}")

## Bước 2: Tạo tập dữ liệu Vi khí hậu & Dị thường Cảm biến
- **Lớp 0 (NORMAL):** Nhiệt độ 20°C - 34°C, Độ ẩm 45% - 80% (Cây trồng phát triển tốt).
- **Lớp 1 (ANOMALY):**
  * Hơ lửa / Nắng nóng cực đoan: Nhiệt độ > 36°C (hoặc lên tới 55°C).
  * Lạnh cóng / Sương giá: Nhiệt độ < 16°C.
  * Khô hạn: Độ ẩm < 35%.
  * Ngập ẩm / Nguy cơ nấm bệnh: Độ ẩm > 88%.

In [ ]:
np.random.seed(42)
N_SAMPLES = 10000

# 1. Sinh dữ liệu Bình thường (NORMAL: 55%)
n_norm = int(N_SAMPLES * 0.55)
temp_norm = np.random.uniform(20.0, 34.0, n_norm) + np.random.normal(0, 0.4, n_norm)
hum_norm  = np.random.uniform(45.0, 80.0, n_norm) + np.random.normal(0, 0.8, n_norm)
labels_norm = np.zeros(n_norm)

# 2. Sinh dữ liệu Bất thường (ANOMALY: 45%)
n_anom = N_SAMPLES - n_norm
n_part = n_anom // 4

# Kịch bản A: Hơ lửa / Nắng gắt
temp_hot = np.random.uniform(36.0, 55.0, n_part)
hum_hot  = np.random.uniform(10.0, 65.0, n_part)

# Kịch bản B: Lạnh buốt
temp_cold = np.random.uniform(0.0, 15.5, n_part)
hum_cold  = np.random.uniform(40.0, 95.0, n_part)

# Kịch bản C: Khô cằn
temp_dry = np.random.uniform(22.0, 36.0, n_part)
hum_dry  = np.random.uniform(5.0, 34.0, n_part)

# Kịch bản D: Ẩm ướt ảnh hưởng cảm biến
temp_wet = np.random.uniform(24.0, 35.0, n_anom - 3 * n_part)
hum_wet  = np.random.uniform(88.0, 99.0, n_anom - 3 * n_part)

temp_anom = np.concatenate([temp_hot, temp_cold, temp_dry, temp_wet])
hum_anom  = np.concatenate([hum_hot, hum_cold, hum_dry, hum_wet])
labels_anom = np.ones(n_anom)

# Gộp và Trộn dữ liệu
X = np.column_stack([
    np.concatenate([temp_norm, temp_anom]),
    np.concatenate([hum_norm, hum_anom])
])
y = np.concatenate([labels_norm, labels_anom])

shuffle_idx = np.random.permutation(len(y))
X, y = X[shuffle_idx], y[shuffle_idx]

# Chia Train/Test 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Tổng mẫu: {len(X)} | Train: {len(X_train)} | Test: {len(X_test)}")

## Bước 3: Trực quan hóa phân bố dữ liệu

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(X[y == 0][:, 0], X[y == 0][:, 1], color='#10b981', alpha=0.35, s=15, label='Normal (Khí hậu chuẩn)')
plt.scatter(X[y == 1][:, 0], X[y == 1][:, 1], color='#ef4444', alpha=0.35, s=15, label='Anomaly (Dị thường / Hơ lửa)')
plt.title('Biểu đồ phân bố Dữ liệu Cảm biến DHT20 (ĐH Bách Khoa - Lab IoT)', fontsize=13)
plt.xlabel('Nhiệt độ (°C)', fontsize=11)
plt.ylabel('Độ ẩm (%)', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper right', frameon=True)
plt.show()

## Bước 4: Xây dựng Mạng nơ-ron (Keras MLP Model)
Cấu trúc tương thích hoàn toàn với vi điều khiển ESP32-S3:
- **Lớp đầu vào:** 2 đặc trưng (`input_1`: `temperature`, `humidity`).
- **Lớp ẩn (Dense):** 8 nơ-ron, hàm kích hoạt `ReLU`.
- **Lớp đầu ra (Dense_1):** 1 nơ-ron, hàm kích hoạt `Sigmoid` (cho ra Anomaly Score từ 0.0 đến 1.0).

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,), name='input_1'),
    tf.keras.layers.Dense(8, activation='relu', name='dense'),
    tf.keras.layers.Dense(1, activation='sigmoid', name='dense_1')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.005),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Bước 5: Huấn luyện mô hình (Training)

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=40,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

## Bước 6: Đánh giá độ chính xác trên tập Test & Ma trận nhầm lẫn

In [ ]:
# Đánh giá Loss & Accuracy
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"✅ Độ chính xác trên tập Test: {test_acc * 100:.2f}%")

# Dự đoán với ngưỡng threshold = 0.60
y_probs = model.predict(X_test)
y_pred = (y_probs >= 0.60).astype(int).flatten()

cm = confusion_matrix(y_test, y_pred)
print("\n--- CONFUSION MATRIX ---")
print(cm)
print("\n--- BÁO CÁO PHÂN LỚP (CLASSIFICATION REPORT) ---")
print(classification_report(y_test, y_pred, target_names=['NORMAL', 'ANOMALY']))

## Bước 7: Chuyển đổi mô hình sang TensorFlow Lite (`.tflite`)
Sử dụng `TFLiteConverter` để nén FlatBuffer dành cho vi điều khiển.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

# Ghi ra file tflite
with open("dht_anomaly_model.tflite", "wb") as f:
    f.write(tflite_model)

print(f"✅ Kích thước file TFLite: {len(tflite_model)} bytes (~{len(tflite_model)/1024:.2f} KB)")

## Bước 8: Xuất file Header C++ (`dht_anomaly_model.h`) cho ESP32-S3
Tự động chuyển các byte nhị phân thành mảng `const unsigned char dht_anomaly_model_tflite[]`.

In [ ]:
def export_c_header(tflite_bytes, filename="dht_anomaly_model.h", var_name="dht_anomaly_model_tflite"):
    c_str = f"/*\n * {filename} - TinyML DHT20 Anomaly Detection Model\n"
    c_str += f" * Tự động sinh từ mô hình Keras / Google Colab (MSSV 2313946)\n"
    c_str += f" * Kích thước: {len(tflite_bytes)} bytes\n */\n\n"
    c_str += "#ifndef DHT_ANOMALY_MODEL_H\n#define DHT_ANOMALY_MODEL_H\n\n"
    c_str += "#include <Arduino.h>\n\n"
    c_str += f"alignas(16) const unsigned char {var_name}[] = {{\n  "
    
    for i, b in enumerate(tflite_bytes):
        c_str += f"0x{b:02x}, "
        if (i + 1) % 12 == 0:
            c_str += "\n  "
            
    c_str = c_str.rstrip(", \n") + "\n};\n\n"
    c_str += f"const unsigned int {var_name}_len = {len(tflite_bytes)};\n\n"
    c_str += "#endif // DHT_ANOMALY_MODEL_H\n"
    
    with open(filename, "w", encoding="utf-8") as f:
        f.write(c_str)
    print(f"🎉 Đã xuất thành công file: {filename} ({len(tflite_bytes)} bytes)")

export_c_header(tflite_model)

# Nếu chạy trên Google Colab, tự động kích hoạt tải file xuống máy tính
try:
    from google.colab import files
    files.download('dht_anomaly_model.h')
    files.download('dht_anomaly_model.tflite')
    print("⬇️ Đã gửi yêu cầu tải file dht_anomaly_model.h về máy bạn!")
except Exception:
    print("Chạy local: File dht_anomaly_model.h đã được lưu trực tiếp tại thư mục hiện tại.")